# Date Occurrence Expansion Test

This notebook expands freeform `event_date` and `event_time` text from `events` into a standardized `occurrences` JSON array and writes it to the `occurrences` JSONB column.

In [ ]:
import json
import os
from datetime import datetime

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field
from sqlalchemy import text
from sqlalchemy.ext.asyncio import create_async_engine

_ = load_dotenv()

TODAY_DATE_TEXT = datetime.now().date().isoformat()
DEFAULT_BATCH_LIMIT = 20

raw_postgres_url = os.environ.get("POSTGRES_URL_2") or os.environ.get("POSTGRES_URL")
if not raw_postgres_url:
    raise ValueError("POSTGRES_URL_2 or POSTGRES_URL must be set")

ASYNC_POSTGRES_URL = raw_postgres_url
if ASYNC_POSTGRES_URL.startswith("postgres://"):
    ASYNC_POSTGRES_URL = ASYNC_POSTGRES_URL.replace("postgres://", "postgresql+asyncpg://", 1)
elif ASYNC_POSTGRES_URL.startswith("postgresql://") and "+asyncpg" not in ASYNC_POSTGRES_URL:
    ASYNC_POSTGRES_URL = ASYNC_POSTGRES_URL.replace("postgresql://", "postgresql+asyncpg://", 1)

research_model = ChatOpenAI(model="gpt-5-mini", temperature=0)
engine = create_async_engine(ASYNC_POSTGRES_URL)

print(f"Today anchor: {TODAY_DATE_TEXT}")

In [ ]:
class EventOccurrenceModelType(BaseModel):
    date: str = Field(description="Occurrence date in YYYY-MM-DD format")
    start_time: str | None = Field(default=None, description="Start time in HH:MM:SS format")
    end_time: str | None = Field(default=None, description="End time in HH:MM:SS format")


class EventOccurrencesExpansionModelType(BaseModel):
    occurrences: list[EventOccurrenceModelType] = Field(default_factory=list)

In [ ]:
DATE_EXPANSION_PROMPT = """
You are an event date/time normalization assistant.

Reference context:
- Today is: {today_date}

Input fields:
- event_name: {event_name}
- event_date (freeform): {event_date}
- event_time (freeform): {event_time}
- event_description (optional): {event_description}

Return format:
- Return a list of objects in this exact shape:
  {{
    "date": "YYYY-MM-DD",
    "start_time": "HH:MM:SS" | null,
    "end_time": "HH:MM:SS" | null
  }}

Rules:
1) Include all date occurrences implied by the text.
2) If a year is missing, infer year using today's date as anchor.
3) Prefer upcoming/current-season interpretation over unsupported historical years.
4) For recurring text, start from the first occurrence on or after today unless source text clearly bounds a historical range.
5) For ranges crossing Dec-Jan, assign the correct year to each day.
6) If time is missing, set start_time and end_time to null.
7) If end time is missing, leave it null.
8) Use 24-hour HH:MM:SS.
9) Do not invent dates not supported by the text.
""".strip()


async def expand_event_occurrences(
    event_name: str | None,
    event_date: str | None,
    event_time: str | None,
    event_description: str | None,
    today_date: str = TODAY_DATE_TEXT,
) -> EventOccurrencesExpansionModelType:
    prompt = DATE_EXPANSION_PROMPT.format(
        today_date=today_date,
        event_name=(event_name or "").strip() or "null",
        event_date=(event_date or "").strip() or "null",
        event_time=(event_time or "").strip() or "null",
        event_description=(event_description or "").strip() or "null",
    )

    structured_model = research_model.with_structured_output(EventOccurrencesExpansionModelType)
    return await structured_model.ainvoke(prompt)

In [ ]:
def _is_valid_date(value: str) -> bool:
    try:
        datetime.strptime(value, "%Y-%m-%d")
        return True
    except ValueError:
        return False


def _is_valid_time_or_none(value: str | None) -> bool:
    if value is None:
        return True
    try:
        datetime.strptime(value, "%H:%M:%S")
        return True
    except ValueError:
        return False


def normalize_occurrences(
    occurrences: list[EventOccurrenceModelType],
) -> list[EventOccurrenceModelType]:
    unique_items: dict[tuple[str, str | None, str | None], EventOccurrenceModelType] = {}

    for occurrence in occurrences:
        if not _is_valid_date(occurrence.date):
            continue
        if not _is_valid_time_or_none(occurrence.start_time):
            continue
        if not _is_valid_time_or_none(occurrence.end_time):
            continue

        dedupe_key = (occurrence.date, occurrence.start_time, occurrence.end_time)
        unique_items[dedupe_key] = occurrence

    return sorted(
        unique_items.values(),
        key=lambda occurrence: (
            occurrence.date,
            occurrence.start_time or "",
            occurrence.end_time or "",
        ),
    )


def occurrences_to_payload(
    occurrences: list[EventOccurrenceModelType],
) -> list[dict[str, str | None]]:
    return [occurrence.model_dump() for occurrence in occurrences]

In [ ]:
async def fetch_events_for_expansion(limit: int = DEFAULT_BATCH_LIMIT) -> list[dict[str, object]]:
    print(f"[fetch] Querying events with occurrences={{}} (limit={limit})")

    query = text(
        """
        SELECT
            id,
            event_name,
            event_date,
            event_time,
            event_description,
            occurrences
        FROM events
        WHERE occurrences = '{}'::jsonb
        ORDER BY created_at DESC
        LIMIT :limit
        """
    )

    async with engine.connect() as connection:
        result = await connection.execute(query, {"limit": limit})
        rows = [dict(row._mapping) for row in result]

    print(f"[fetch] Retrieved {len(rows)} row(s)")
    return rows


async def update_event_occurrences_jsonb(
    event_id: str,
    occurrences_payload: list[dict[str, str | None]],
) -> None:
    print(
        f"[update] Writing {len(occurrences_payload)} occurrence(s) for event_id={event_id}"
    )

    update_statement = text(
        """
        UPDATE events
        SET occurrences = CAST(:occurrences_json AS jsonb)
        WHERE id = CAST(:event_id AS uuid)
          AND occurrences = '{}'::jsonb
        """
    )

    async with engine.begin() as connection:
        update_result = await connection.execute(
            update_statement,
            {
                "occurrences_json": json.dumps(occurrences_payload),
                "event_id": event_id,
            },
        )

    if update_result.rowcount == 0:
        raise ValueError(f"Event {event_id} was not updated because occurrences was not empty JSON")

    print(f"[update] Success for event_id={event_id}")

In [ ]:
async def apply_date_expansion_updates(limit: int = DEFAULT_BATCH_LIMIT) -> list[str]:
    print(f"[apply] Starting updates (limit={limit})")

    rows = await fetch_events_for_expansion(limit=limit)
    updated_ids: list[str] = []

    for row_index, row in enumerate(rows, start=1):
        event_id = str(row["id"])
        event_name = str(row.get("event_name") or "")
        print(f"[apply] {row_index}/{len(rows)} processing event_id={event_id} name={event_name!r}")

        expansion = await expand_event_occurrences(
            event_name=event_name,
            event_date=str(row.get("event_date") or ""),
            event_time=str(row.get("event_time") or ""),
            event_description=str(row.get("event_description") or ""),
        )

        normalized = normalize_occurrences(expansion.occurrences)
        payload = occurrences_to_payload(normalized)

        print(
            f"[apply] event_id={event_id} raw={len(expansion.occurrences)} normalized={len(normalized)}"
        )

        await update_event_occurrences_jsonb(
            event_id=event_id,
            occurrences_payload=payload,
        )
        updated_ids.append(event_id)

    print(f"[apply] Completed. Updated {len(updated_ids)} row(s)")
    return updated_ids



In [ ]:

# Run only after reviewing dry run output:
updated_event_ids = await apply_date_expansion_updates(limit=20)
updated_event_ids[:5]